# 02 - GDELT Property Graph Creation

**Execution Order: Run this notebook SECOND**

This notebook creates the BigQuery Property Graph from the prepared GDELT data.

## Workflow
1. ⏮️ Previous: `01_gdelt_data_prep.ipynb` - Prepares data and creates tables
2. ✅ **This notebook** - Creates the property graph

## Prerequisites
- Run `01_gdelt_data_prep.ipynb` first to prepare the data tables
- Ensure all required tables exist in your BigQuery dataset

**Optional later steps** (do not replace this notebook):

- `03_gdelt_incremental_refresh.ipynb` — incremental GDELT merges and person-table procedure calls after you have a working warehouse from notebook 01.
- `04_gdelt_data_profiling.ipynb` — Dataplex profiling and descriptions for tables in your dataset.


## 🔧 Configuration Setup

This cell loads configuration from `config.py`:

### **Project Settings**
- **GCP_PROJECT_ID**: Your Google Cloud Platform project identifier
- **BIGQUERY_DATASET**: Dataset name containing the prepared GDELT tables

⚠️ **Important**: Update `GCP_PROJECT_ID` in `config.py` with your actual project ID before running (or set `GDELT_GRAPH_DEMO_GCP_PROJECT`; see `config.py`). These notebooks intentionally do **not** default from `GOOGLE_CLOUD_PROJECT`.


In [ ]:
# Import configuration from config.py
from config import (
    GCP_PROJECT_ID,
    BIGQUERY_DATASET,
    print_config,
    validate_config
)

# Display configuration
print_config()

# Validate configuration
if not validate_config():
    print("\n⚠️  Please update config.py before proceeding!")
else:
    print("\n✅ Configuration validated successfully!")


## 📚 Library Imports

This cell imports all necessary Python libraries for the GDELT data analysis workflow:

### **Google Cloud Services**
- `google.cloud.bigquery`: For querying and managing BigQuery data
- `google.cloud.storage`: Imported for parity with notebook 01; **not used** in this notebook’s cells (safe to remove from code if you prefer a minimal import list)
- `google.auth`: For GCP authentication handling

### **Data Processing**
- `pandas`: For data manipulation and analysis
- `json`: For JSON data handling
- `datetime`: For date/time operations

### **Network Analysis & Visualization**
- `networkx`: For creating and analyzing graph networks
- `matplotlib.pyplot`: For creating visualizations and plots

### **System & File Operations**
- `os`, `pathlib.Path`: For file system operations
- `subprocess`: For running system commands
- `shutil`: For file operations

All libraries are tested for successful import with confirmation messages.

In [ ]:
# Import required libraries
import os
import pandas as pd
from google.cloud import bigquery
from google.cloud import storage
import json
from datetime import datetime
import networkx as nx
import matplotlib.pyplot as plt
import os
from pathlib import Path
import subprocess
import os
import shutil
from google.auth import default
from google.auth.exceptions import DefaultCredentialsError
from google.cloud import bigquery
from datetime import datetime

print("✅ All libraries imported successfully!")
print("   - BigQuery and Cloud Storage clients ready")
print("   - NetworkX and Matplotlib ready for visualization")
print("   - Pandas ready for data processing")


## 🔐 GCP Authentication Setup

This cell provides a comprehensive GCP authentication function that handles various authentication scenarios:

### **Authentication Process**
1. **Credential Check**: Verifies existing Google Cloud credentials
2. **Project Validation**: Ensures credentials match the target project
3. **Credential Reset**: Clears old credentials if project mismatch detected
4. **Project Configuration**: Sets the correct GCP project using gcloud CLI
5. **Re-authentication**: Initiates browser-based OAuth flow if needed
6. **Quota Project**: Sets quota project to avoid billing warnings
7. **Verification**: Confirms successful authentication

### **Error Handling**
- Handles missing credentials gracefully
- Provides manual fallback instructions
- Manages project mismatches automatically
- Shows detailed error messages and troubleshooting tips

### **Environment Setup**
- Sets `GOOGLE_CLOUD_PROJECT` environment variable
- Configures application default credentials
- Prepares credentials for BigQuery and Cloud Storage clients

⚡ **Note**: This function may open a browser window for OAuth authentication.


In [ ]:
# GCP Authentication Setup


def setup_gcp_authentication():
    """Complete GCP authentication setup with error handling"""
    print("🔐 Setting up GCP Authentication...")
    
    try:
        # Step 1: Try to use existing credentials first
        print("🔍 Checking for existing credentials...")
        try:
            credentials, default_project = default()
            print(f"✅ Found existing credentials for project: {default_project}")
            
            # If the project matches, we're good
            if default_project == GCP_PROJECT_ID:
                print(f"🎯 Project matches target project: {GCP_PROJECT_ID}")
                os.environ['GOOGLE_CLOUD_PROJECT'] = GCP_PROJECT_ID
                return credentials, GCP_PROJECT_ID
            else:
                print(f"⚠️  Project mismatch: {default_project} vs {GCP_PROJECT_ID}")
                print("🔄 Will re-authenticate with correct project...")
        except DefaultCredentialsError:
            print("❌ No existing credentials found")
            print("🔄 Will authenticate from scratch...")
        
        # Step 2: Clear old credentials if needed
        print("🗑️  Clearing old credentials...")
        adc_path = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
        if os.path.exists(adc_path):
            os.remove(adc_path)
            print("✅ Removed old application default credentials")
        
        # Step 3: Set the correct project
        print(f"🎯 Setting gcloud project to: {GCP_PROJECT_ID}")
        result = subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT_ID], 
                              capture_output=True, text=True, check=True)
        print("✅ Project set successfully")
        
        # Step 4: Re-authenticate
        print("🔄 Re-authenticating with application default credentials...")
        print("   This will open a browser window for authentication...")
        
        result = subprocess.run(['gcloud', 'auth', 'application-default', 'login'], 
                              check=True)
        print("✅ Re-authentication successful")
        
        # Step 5: Set quota project to avoid warnings
        print("💰 Setting quota project...")
        try:
            subprocess.run(['gcloud', 'auth', 'application-default', 'set-quota-project', GCP_PROJECT_ID], 
                          capture_output=True, text=True, check=True)
            print("✅ Quota project set successfully")
        except:
            print("⚠️  Could not set quota project (this is usually fine)")
        
        # Step 6: Verify the setup
        print("🧪 Verifying authentication...")
        credentials, project = default()
        print(f"✅ Authentication successful - Project: {project}")
        
        # Set environment variable
        os.environ['GOOGLE_CLOUD_PROJECT'] = GCP_PROJECT_ID
        print(f"🌍 Set GOOGLE_CLOUD_PROJECT environment variable to: {GCP_PROJECT_ID}")
        
        return credentials, GCP_PROJECT_ID
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Command failed: {e}")
        print("💡 Manual steps required:")
        print(f"   1. gcloud config set project {GCP_PROJECT_ID}")
        print("   2. gcloud auth application-default login")
        print(f"   3. gcloud auth application-default set-quota-project {GCP_PROJECT_ID}")
        return None, None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None, None

# Run authentication setup
credentials, authenticated_project = setup_gcp_authentication()


## 🕸️ Create Property Graph

This cell executes the DDL to create a Property Graph in BigQuery.

### **Graph Structure**
#### **Nodes**
- **Person**: Derived from `person` table
- **Organization**: Derived from `organization` table
- **Location**: Derived from `location` table
- **Event**: Derived from `event` table
- **Article**: Derived from `article` table

#### **Edges**
- **CO_OCCURS_WITH**: Person to Person (from `person_cooccurrence`)
- **AFFILIATED_WITH**: Person to Organization (from `person_organization`)
- **LOCATED_AT**: Person to Location (from `person_location`)

### **Logic**
1. Initializes the BigQuery client.
2. Defines the `CREATE OR REPLACE PROPERTY GRAPH` SQL statement.
3. Executes the query to build the complete graph schema.


In [ ]:
# Create Graph
from google.api_core.exceptions import GoogleAPICallError

# Initialize BigQuery client
client = bigquery.Client(project=GCP_PROJECT_ID, credentials=credentials)

# Define the CREATE PROPERTY GRAPH query.
#
# Notes:
# - All five node tables are exposed (person, organization, location, event,
#   article). `article` has no edge tables connecting to it yet, but is
#   declared so future edges (e.g. mentions_article) require no DDL change to
#   the graph itself.
# - `event_participant` is intentionally OMITTED from the graph: its
#   participant_id column is polymorphic (PERSON / ORGANIZATION / LOCATION
#   driven by participant_type). GoogleSQL property graph DDL requires each
#   edge to declare a single REFERENCES target node table, so a polymorphic
#   FK cannot be modelled without splitting the table into per-type edges.
#   Track that as a follow-up; for now event participation is queryable via
#   plain SQL only.
# - PROPERTIES ARE ALL COLUMNS exposes every column to GQL (including
#   created_at / updated_at / name_variations etc.) so MATCH queries can
#   reference any field without re-running this DDL.
create_graph_query = f"""
CREATE OR REPLACE PROPERTY GRAPH {BIGQUERY_DATASET}.GdeltGraph
  NODE TABLES (
    {BIGQUERY_DATASET}.person
      KEY (person_id)
      LABEL Person PROPERTIES ARE ALL COLUMNS,
    {BIGQUERY_DATASET}.organization
      KEY (org_id)
      LABEL Organization PROPERTIES ARE ALL COLUMNS,
    {BIGQUERY_DATASET}.location
      KEY (location_id)
      LABEL Location PROPERTIES ARE ALL COLUMNS,
    {BIGQUERY_DATASET}.event
      KEY (event_id)
      LABEL Event PROPERTIES ARE ALL COLUMNS,
    {BIGQUERY_DATASET}.article
      KEY (article_id)
      LABEL Article PROPERTIES ARE ALL COLUMNS
  )
  EDGE TABLES (
    {BIGQUERY_DATASET}.person_cooccurrence
      KEY (relationship_id)
      SOURCE KEY (person1_id) REFERENCES person (person_id)
      DESTINATION KEY (person2_id) REFERENCES person (person_id)
      LABEL CO_OCCURS_WITH PROPERTIES ARE ALL COLUMNS,
    {BIGQUERY_DATASET}.person_organization
      KEY (relationship_id)
      SOURCE KEY (person_id) REFERENCES person (person_id)
      DESTINATION KEY (org_id) REFERENCES organization (org_id)
      LABEL AFFILIATED_WITH PROPERTIES ARE ALL COLUMNS,
    {BIGQUERY_DATASET}.person_location
      KEY (relationship_id)
      SOURCE KEY (person_id) REFERENCES person (person_id)
      DESTINATION KEY (location_id) REFERENCES location (location_id)
      LABEL LOCATED_AT PROPERTIES ARE ALL COLUMNS
  );
"""

# Execute the query
print(f"Creating Property Graph '{BIGQUERY_DATASET}.GdeltGraph'...")
try:
    job = client.query(create_graph_query)
    job.result()  # Wait for the job to complete
    print(f"✅ Property Graph '{BIGQUERY_DATASET}.GdeltGraph' created successfully.")
except GoogleAPICallError as e:
    print(f"❌ Error creating graph: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

In [ ]:
# Graph smoke test + optional filtered query
#
# Why the old Nordic-style example often returns 0 rows:
# - Geo columns come from GKG V2Locations (split by '#'): countries[2], states/ADM1[3],
#   cities[1] only when location type is 3 or 4. Codes often differ from guesses
#   (e.g. ADM1 may be 'NO21' not 'NO-21'; Norway is usually 'NO' not 'SJ').
# - Many edges have empty countries/states/cities arrays if articles lack locations.
# - cooccurrence_count > 20 is very selective unless your gkg_partitioned window is large.
#
# If MATCH returns 0 rows while person_cooccurrence still has rows, see (1b): BigQuery only
# traverses CO_OCCURS_WITH when person1_id and person2_id both exist in `person`. Rebuilding
# `person` issues new UUIDs — rerun sp_update_person_cooccurrence_table(NULL) afterward.

smoke_query = f"""
GRAPH {BIGQUERY_DATASET}.GdeltGraph
MATCH (p1:Person)-[co:CO_OCCURS_WITH]->(p2:Person)
RETURN p1.name AS p1, p2.name AS p2, co.cooccurrence_count AS cnt,
       co.countries AS countries, co.states AS states, co.cities AS cities
LIMIT 20;
"""

diag_sql = f"""
SELECT
  COUNT(*) AS total_edges,
  COUNTIF(cooccurrence_count > 20) AS cnt_gt_20,
  COUNTIF(ARRAY_LENGTH(coalesce(cities, [])) > 0) AS any_city,
  COUNTIF('London' IN UNNEST(coalesce(cities, []))) AS city_london,
  COUNTIF('NO-21' IN UNNEST(coalesce(states, []))) AS state_no21,
  COUNTIF('SJ' IN UNNEST(coalesce(countries, []))) AS country_sj,
  COUNTIF(
    cooccurrence_count > 20
    AND (
      'NO-21' IN UNNEST(coalesce(states, []))
      OR 'SJ' IN UNNEST(coalesce(countries, []))
      OR 'London' IN UNNEST(coalesce(cities, []))
    )
  ) AS matches_old_example_filter
FROM `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.person_cooccurrence`;
"""

referential_integrity_sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.person`) AS person_table_rows,
  COUNT(*) AS total_edges,
  COUNTIF(p1.person_id IS NOT NULL) AS person1_exists_in_person_table,
  COUNTIF(p2.person_id IS NOT NULL) AS person2_exists_in_person_table,
  COUNTIF(p1.person_id IS NOT NULL AND p2.person_id IS NOT NULL) AS both_endpoints_in_person
FROM `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.person_cooccurrence` c
LEFT JOIN `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.person` p1 ON c.person1_id = p1.person_id
LEFT JOIN `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.person` p2 ON c.person2_id = p2.person_id;
"""

filtered_graph_query = f"""
GRAPH {BIGQUERY_DATASET}.GdeltGraph
MATCH (p1:Person)-[co1:CO_OCCURS_WITH]->(p2:Person)
WHERE co1.cooccurrence_count >= 5
RETURN p1.name AS p1, p2.name AS p2, co1.cooccurrence_count AS cnt,
       co1.themes AS themes, co1.countries AS countries, co1.states AS states,
       co1.cities AS cities
LIMIT 50;
"""

print("(1) Standard SQL diagnostic on person_cooccurrence (explains empty MATCH filters)...")
try:
    diag_df = client.query(diag_sql).to_dataframe()
    display(diag_df)
except GoogleAPICallError as e:
    print(f"❌ Diagnostic query failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

print(
    "\n(1b) Endpoint integrity — CO_OCCURS_WITH only traverses when both IDs exist in `person`..."
)
try:
    ri_df = client.query(referential_integrity_sql).to_dataframe()
    display(ri_df)
except GoogleAPICallError as e:
    print(f"❌ Referential diagnostic failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

print("\n(2) Graph smoke test — any CO_OCCURS_WITH edges...")
try:
    smoke_df = client.query(smoke_query).to_dataframe()
    if len(smoke_df) == 0:
        print(
            "⚠️ Smoke test: 0 rows. Property-graph MATCH needs both endpoints in `person`. "
            "If (1b) person_table_rows is 0, run CALL "
            f"`{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.sp_update_person_table`(NULL) first; "
            "then CALL "
            f"`{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.sp_update_person_cooccurrence_table`(NULL). "
            "Use location US-CENTRAL1 (match your dataset). If person_table_rows > 0 but "
            "both_endpoints_in_person is 0, cooccurrence still has stale UUIDs — rerun only "
            "the cooccurrence CALL."
        )
    else:
        print(f"✅ Smoke test: {len(smoke_df)} rows.")
    display(smoke_df)
except GoogleAPICallError as e:
    print(f"❌ Error running smoke graph query: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

print("\n(3) Filtered graph query — strong pairs with some metadata (adjust threshold as needed)...")
try:
    results_df = client.query(filtered_graph_query).to_dataframe()
    if len(results_df) == 0:
        print(
            "⚠️ Filtered query: 0 rows — often the same endpoint/graph issue as the smoke test "
            "(not only strict filters)."
        )
    else:
        print(f"✅ Filtered query: {len(results_df)} rows.")
    display(results_df)
except GoogleAPICallError as e:
    print(f"❌ Error running filtered graph query: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")